# Tiny VLA — Teach a Robot with Language 🤖
**AI Builders Lab | TensorFlow | Google Colab**

Learn how a small **Vision-Language-Action** model turns a camera image and an instruction into robot gripper motion. The policy is trained on synthetic demonstrations, saved, then **reloaded** and used to control an animated 2D robot. Run **Runtime → Run all**; CPU is enough.

**Scope:** this is a *trained-from-scratch educational mini-VLA*, not a downloaded pretrained robotics foundation model. Its 2D gripper simulator is intentionally simplified (no rigid-body contact physics).

## 1. What is a VLA?
Input: 64×64 RGB camera image + a text command.
Output: motion and grasp commands: `[dx, dy, gripper]`.

The robot action policy is learned from demonstrations: **vision CNN + language embedding → action predictor**.

The teacher controller is used only for **training-label generation**, never as a fallback at inference. This is a small neural multimodal policy, not a Transformer.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML

SEED = 17
tf.keras.utils.set_random_seed(SEED)
SIZE = 48
COLORS = {"red":(0.95,.2,.2),"blue":(.15,.4,.95),"green":(.12,.75,.32)}
NAMES = list(COLORS)
COMMANDS = [f"{verb} the {color} block" for verb in ["touch","pick"] for color in NAMES]
STEP = .09
RADIUS = .11
TRAIN_SAMPLES = 5000
EPOCHS = 16
print("Commands:",COMMANDS)

## 2. Build a tiny camera simulator
We draw colored blocks and a robot gripper on a tabletop. Each scene contains randomly placed objects. For simplicity, the simulated robot moves directly in the plane. The learned policy receives **pixels only**, not hidden target coordinates.

In [ ]:
def make_scene(rng):
    while True:
        blocks=rng.uniform(.15,.85,(3,2)).astype("float32")
        if min(np.linalg.norm(blocks[i]-blocks[j]) for i in range(3) for j in range(i+1,3))>.20: break
    return {"blocks":blocks, "robot":rng.uniform(.1,.9,2).astype("float32"),"holding":-1}

def camera(scene):
    img=np.full((SIZE,SIZE,3),.94,dtype="float32")
    for i,name in enumerate(NAMES):
        x,y=scene["blocks"][i]
        px,py=int(x*(SIZE-1)),int((1-y)*(SIZE-1))
        img[max(py-3,0):min(py+4,SIZE),max(px-3,0):min(px+4,SIZE)] = COLORS[name]
    x,y=scene["robot"]
    px,py=int(x*(SIZE-1)),int((1-y)*(SIZE-1))
    img[max(py-2,0):min(py+3,SIZE),max(px-2,0):min(px+3,SIZE)] = (.02,.02,.02)
    return img

rng=np.random.default_rng(SEED)
demo=make_scene(rng)
plt.figure(figsize=(4,4));plt.imshow(camera(demo));plt.axis("off");plt.title("Robot camera");plt.show()

## 3. Language tokens and expert actions
The expert controller has access to object coordinates for generating training labels only. The actual neural policy sees only the rendered image and text.

A command `touch the red block` means move near it; `pick the blue block` means approach and close the gripper.

In [ ]:
VOCAB={w:i+1 for i,w in enumerate(sorted(set(" ".join(COMMANDS).split())))}
def tokenize(command):
    command=command.lower().strip()
    if command not in COMMANDS: raise ValueError("Choose from: "+str(COMMANDS))
    ids=[VOCAB[w] for w in command.split()]
    return np.array(ids+[0]*(4-len(ids)),dtype="int32")

def goal(command):
    return next(i for i,name in enumerate(NAMES) if name in command), command.startswith("pick")

def expert(scene,command):
    i,pick=goal(command)
    diff=scene["blocks"][i]-scene["robot"]
    move=np.clip(diff/STEP,-1,1)
    grip=float(pick and np.linalg.norm(diff)<RADIUS)
    return np.array([move[0],move[1],grip],dtype="float32")

def execute(scene,action,command):
    scene["robot"]=np.clip(scene["robot"]+np.clip(action[:2],-1,1)*STEP,.04,.96)
    i,pick=goal(command)
    if pick and action[2]>.5 and np.linalg.norm(scene["robot"]-scene["blocks"][i])<RADIUS:
        scene["holding"]=i
    if scene["holding"]>=0:scene["blocks"][scene["holding"]]=scene["robot"].copy()
    return scene

print("Token example:",tokenize("pick the blue block"))
print("Action example:",expert(demo,"pick the blue block"))

## 4. Generate demonstrations
Random tabletop scenes provide image, instruction, and desired expert action examples. Some training samples position the gripper near the block so the model learns when to grasp.

In [ ]:
def dataset(n,seed):
    rng=np.random.default_rng(seed)
    images=np.empty((n,SIZE,SIZE,3),dtype="float32")
    words=np.empty((n,4),dtype="int32")
    actions=np.empty((n,3),dtype="float32")
    for k in range(n):
        scene=make_scene(rng)
        cmd=COMMANDS[int(rng.integers(len(COMMANDS)))]
        i,_=goal(cmd)
        if rng.random()<.42:
            scene["robot"]=np.clip(scene["blocks"][i]+rng.normal(0,.07,2),.04,.96).astype("float32")
        images[k]=camera(scene)
        words[k]=tokenize(cmd)
        actions[k]=expert(scene,cmd)
    return images,words,actions

train_images,train_words,train_actions=dataset(TRAIN_SAMPLES,SEED+1)
test_images,test_words,test_actions=dataset(600,SEED+2)
print(train_images.shape,train_words.shape,train_actions.shape)

## 5. Train a neural VLA policy
The image CNN extracts visual features. The language embedding encodes the words. Concatenation fuses visual and language information; dense layers predict planar motion and a grasp probability.

We use **imitation learning** (supervised behavior cloning), not reinforcement learning.

In [ ]:
image_input=tf.keras.Input((SIZE,SIZE,3),name="image")
language_input=tf.keras.Input((4,),dtype="int32",name="language")
v=tf.keras.layers.Conv2D(16,5,strides=2,activation="relu")(image_input)
v=tf.keras.layers.Conv2D(32,3,strides=2,activation="relu")(v)
v=tf.keras.layers.Conv2D(32,3,strides=2,activation="relu")(v)
v=tf.keras.layers.Flatten()(v)
v=tf.keras.layers.Dense(96,activation="relu")(v)
l=tf.keras.layers.Embedding(len(VOCAB)+1,12)(language_input)
l=tf.keras.layers.Flatten()(l)
l=tf.keras.layers.Dense(48,activation="relu")(l)
h=tf.keras.layers.Concatenate()([v,l])
h=tf.keras.layers.Dense(128,activation="relu")(h)
movement=tf.keras.layers.Dense(2,activation="tanh",name="movement")(h)
grasp=tf.keras.layers.Dense(1,activation="sigmoid",name="grasp")(h)
policy=tf.keras.Model([image_input,language_input],[movement,grasp])
policy.compile(optimizer=tf.keras.optimizers.Adam(.001),
               loss={"movement":"mse","grasp":"binary_crossentropy"},
               loss_weights={"movement":1,"grasp":.3})
history=policy.fit({"image":train_images,"language":train_words},
                   {"movement":train_actions[:,:2],"grasp":train_actions[:,2:]},
                   validation_data=({"image":test_images,"language":test_words},
                                    {"movement":test_actions[:,:2],"grasp":test_actions[:,2:]}),
                   epochs=EPOCHS,batch_size=128,verbose=1)
plt.plot(history.history["loss"],label="Training")
plt.plot(history.history["val_loss"],label="Validation")
plt.xlabel("Epoch");plt.ylabel("Loss");plt.legend();plt.grid(alpha=.2);plt.show()

## 6. Save and LOAD the Mini-VLA model
We save the trained neural network as `tiny_vla.keras`, then reload it from disk. The next experiment controls the robot using this **loaded model**, with no expert actions.

In [ ]:
policy.save("tiny_vla.keras")
vla=tf.keras.models.load_model("tiny_vla.keras")
def act(scene,command):
    image=camera(scene)[None,...]
    word=tokenize(command)[None,...]
    move,grip=vla.predict({"image":image,"language":word},verbose=0)
    return np.concatenate([move[0],grip[0]])
print("Reloaded mini VLA")
print("Example action:",act(demo,"touch the red block").round(3))

## 7. Watch language become robot actions
Change `COMMAND` to any of the six allowed sentences. The policy repeatedly predicts actions, which the simulator executes. The visualization shows its behavior step-by-step.

In [ ]:
COMMAND="pick the blue block"
SIM_STEPS=24
scene=make_scene(np.random.default_rng(SEED+21))
snapshots=[]
actions=[]
for t in range(SIM_STEPS):
    snapshots.append((scene["robot"].copy(),scene["blocks"].copy(),scene["holding"]))
    action=act(scene,COMMAND)
    actions.append(action.copy())
    execute(scene,action,COMMAND)
snapshots.append((scene["robot"].copy(),scene["blocks"].copy(),scene["holding"]))
target,pick=goal(COMMAND)
distance=np.linalg.norm(scene["robot"]-scene["blocks"][target])
success=scene["holding"]==target if pick else distance<RADIUS
print("Instruction:",COMMAND,"Success:",success,"Final distance:",round(float(distance),3))
fig,ax=plt.subplots(figsize=(6,6))
def draw(t):
    ax.clear()
    ax.set_xlim(0,1);ax.set_ylim(0,1);ax.set_aspect("equal")
    ax.set_title(f"{COMMAND} | step {t}/{SIM_STEPS}")
    robot,blocks,holding=snapshots[t]
    for i,name in enumerate(NAMES):
        ax.add_patch(Rectangle(blocks[i]-.035,.07,.07,color=COLORS[name]))
        ax.text(blocks[i,0],blocks[i,1]+.055,name,ha="center",fontsize=10)
    ax.add_patch(Circle(robot,.028,color="black"))
    if t<len(actions):
        vec=actions[t][:2]*STEP*1.6
        ax.arrow(robot[0],robot[1],vec[0],vec[1],width=.003,color="orange")
    ax.text(.03,.03,"Gripper: "+("CLOSED" if holding>=0 else "OPEN"))
    ax.set_xticks([]);ax.set_yticks([])
anim=FuncAnimation(fig,draw,frames=len(snapshots),interval=220,repeat=False)
plt.close(fig)
display(HTML(anim.to_jshtml()))

## 8. Explore and evaluate
Try different commands, scene seeds, sample sizes, training epochs, and robot step sizes. The learned model may fail—this is an important part of the lesson.

**Questions:** Why does the robot need a camera? What role do word embeddings play? Why isn't this a general-purpose VLA? How would real gripper kinematics, collisions, or unseen vocabulary change the problem?

**Key distinction:** A VLA learns **what action to take**; a World Model predicts **what happens next**.

In [ ]:
for command in COMMANDS:
    rng=np.random.default_rng(303)
    wins=0
    trials=5
    for _ in range(trials):
        scene=make_scene(rng)
        for t in range(24): execute(scene,act(scene,command),command)
        i,pick=goal(command)
        dist=np.linalg.norm(scene["robot"]-scene["blocks"][i])
        wins+=int(scene["holding"]==i if pick else dist<RADIUS)
    print(f"{command:24s}: {wins}/{trials} successes")